# Inflation (CPI) — Research Workbench

Research mirror of `sections/inflation.py`. Same code path as the Streamlit app — no duplicated logic.

**Design note**: YoY and 3m-annualized are computed on the FULL history, then the display window is filtered after. This avoids losing the first 12 months of real YoY data.

## Setup (standard 5 cells — same for every section notebook)

In [ ]:
!pip install fredapi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
import os
PROJECT_PATH = "/content/drive/MyDrive/FREDMACRO"

if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

In [ ]:
os.environ["FRED_API_KEY"] = "6e079bc3e1ab2b8280b94e05ff432f30"

In [ ]:
# Sanity: confirm the shared modules import cleanly
from core import config, transforms
from core.fred_client import get_series
from sections import inflation

print('config.DEFAULT_START_DATE =', config.DEFAULT_START_DATE)
print('config.CACHE_DIR =', config.CACHE_DIR)
print('inflation.CPI_DEFAULT_START =', inflation.CPI_DEFAULT_START)

## Pattern A — Call individual chart functions

Each chart is its own function in `sections.inflation` and returns a `plotly.graph_objects.Figure`.

In [ ]:
# Combined 3-panel dashboard (matches the original Colab function)
inflation.cpi_dashboard(start_date='2015-08-01').show()

In [ ]:
inflation.cpi_headline_yoy(start_date='2015-08-01').show()

In [ ]:
inflation.cpi_services_breakdown(start_date='2015-08-01').show()

In [ ]:
inflation.cpi_momentum(start_date='2015-08-01').show()

## Pattern B — Inspect the underlying frames

The transforms are pure — call `yoy_change` / `annualized_change` directly to see exactly what's feeding each chart.

In [ ]:
# Headline CPI YoY — last 12 months
cpi = get_series('CPIAUCSL')
yoy = transforms.yoy_change(cpi)
(yoy.tail(12) * 100).round(2)

In [ ]:
# 3m annualized headline vs core, last 6 months
import pandas as pd
frame = pd.DataFrame({
    'CPI 3m ann':      transforms.annualized_change(get_series('CPIAUCSL'), periods=3),
    'Core CPI 3m ann': transforms.annualized_change(get_series('CPILFESL'), periods=3),
})
(frame.tail(6) * 100).round(2)

## Pattern C — Build the whole section like Streamlit does

In [ ]:
section = inflation.build(start_date='2015-08-01')

print(section['title'])
for chart in section['charts']:
    print(f"  - {chart['id']}: {chart['title']}")

In [ ]:
for chart in section['charts']:
    print(chart['title'])
    chart['fig'].show()
    if chart.get('commentary'):
        print('   →', chart['commentary'])
    print()